# OLMo2 ProFLingo Reference Robustness

Compare ProFLingo `match_rate` across OLMo2 derivatives for each available reference fingerprint. In the project's robustness framing, each subplot estimates how the match rate changes as we move away from that reference model's position in the lineage.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

## Configuration

`REFERENCE_REPORTS` is the active set of comparable ProFLingo trajectory reports. `REFERENCE_TARGET_KEYS` tells the plot where the reference model itself lies on the x-axis. To add the base reference once it is normalized, move the `Base` entry from `OPTIONAL_REFERENCE_REPORTS` into `REFERENCE_REPORTS` and add its target key.

In [ ]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "verification"

REFERENCE_REPORTS = {
    "Base reference": ARTIFACT_DIR / "olmo2_base_proflingo_reference_trajectory.json",
    "SFT reference": ARTIFACT_DIR / "olmo2_sft_proflingo_reference_trajectory.json",
    "DPO reference": ARTIFACT_DIR / "olmo2_dpo_proflingo_reference_trajectory.json",
    "RLVR reference": ARTIFACT_DIR / "olmo2_rlvr1_proflingo_reference_trajectory.json",
    "Instruct reference": ARTIFACT_DIR / "olmo2_instruct_proflingo_reference_trajectory.json",
}

REFERENCE_TARGET_KEYS = {
    "Base reference": "base@main",
    "SFT reference": "sft@main",
    "DPO reference": "dpo@main",
    "RLVR reference": "rlvr1@main",
    "Instruct reference": "instruct@main",
}

ALL_REFERENCE_TARGET_KEYS = REFERENCE_TARGET_KEYS

MODEL_VERSION_ORDER = [
    ("base@main", "base"),
    ("sft@main", "SFT"),
    ("dpo@main", "DPO"),
    ("rlvr1@step_400", "RLVR 400"),
    ("rlvr1@step_800", "RLVR 800"),
    ("rlvr1@step_1200", "RLVR 1200"),
    ("rlvr1@step_1600", "RLVR 1600"),
    ("rlvr1@step_2000", "RLVR 2000"),
    ("rlvr1@step_2400", "RLVR 2400"),
    ("rlvr1@main", "RLVR main"),
    ("instruct@step_400", "Instruct 400"),
    ("instruct@step_800", "Instruct 800"),
    ("instruct@step_1200", "Instruct 1200"),
    ("instruct@step_1600", "Instruct 1600"),
    ("instruct@step_2000", "Instruct 2000"),
    ("instruct@step_2400", "Instruct 2400"),
    ("instruct@main", "Instruct Main"),
]

## Load Reports

The parser only depends on the `proflingo[target_key].match_rate` summary and the shared target keys above, so adding another normalized reference report should require changing only `REFERENCE_REPORTS`.

In [ ]:
def load_json(path: Path) -> dict:
    with path.open() as f:
        return json.load(f)


def extract_proflingo_match_rates(reference_name: str, path: Path) -> list[dict]:
    report = load_json(path)
    proflingo = report.get("proflingo") or {}
    rows = []
    for order, (target_key, model_version) in enumerate(MODEL_VERSION_ORDER):
        result = proflingo.get(target_key) or {}
        rows.append(
            {
                "reference": reference_name,
                "reference_model": report.get("reference_model"),
                "report_path": str(path),
                "target_key": target_key,
                "model_version": model_version,
                "order": order,
                "matched": result.get("matched"),
                "total": result.get("total"),
                "match_rate": result.get("match_rate"),
                "is_reference_target": target_key == ALL_REFERENCE_TARGET_KEYS.get(reference_name),
            }
        )
    return rows


rows = []
missing_reports = []
for reference_name, report_path in REFERENCE_REPORTS.items():
    if report_path.exists():
        rows.extend(extract_proflingo_match_rates(reference_name, report_path))
    else:
        missing_reports.append(report_path)

match_df = pd.DataFrame(rows)

if missing_reports:
    print("Missing reports:")
    for report_path in missing_reports:
        print(f"- {report_path}")

print(f"Loaded {match_df['reference'].nunique()} reference reports")
print(f"Rows: {len(match_df)}")
match_df.head()

## Robustness Plots

Each subplot marks the reference model's own x-axis position with a dashed vertical line and a larger diamond marker. That makes it easier to read the left/right falloff as the lineage moves away from the reference.

In [ ]:
reference_names = list(REFERENCE_REPORTS)
fig, axes = plt.subplots(
    nrows=len(reference_names),
    ncols=1,
    figsize=(15, max(3.2 * len(reference_names), 6)),
    sharex=True,
    sharey=True,
)
if len(reference_names) == 1:
    axes = [axes]

x_labels = [label for _, label in MODEL_VERSION_ORDER]
for ax, reference_name in zip(axes, reference_names):
    group = match_df[match_df["reference"] == reference_name].sort_values("order")
    ax.plot(
        group["order"],
        group["match_rate"],
        marker="o",
        linewidth=2,
        color="tab:blue",
    )

    reference_point = group[group["is_reference_target"]]
    if not reference_point.empty:
        ref_row = reference_point.iloc[0]
        ax.axvline(ref_row["order"], color="tab:red", linestyle="--", linewidth=1.5, alpha=0.8)
        ax.scatter(
            [ref_row["order"]],
            [ref_row["match_rate"]],
            marker="D",
            s=90,
            color="tab:red",
            edgecolor="white",
            linewidth=1,
            zorder=4,
        )
        ax.annotate(
            "reference",
            xy=(ref_row["order"], ref_row["match_rate"]),
            xytext=(6, 12),
            textcoords="offset points",
            color="tab:red",
            fontsize=9,
        )
    else:
        ax.text(0.01, 0.92, "reference position not configured", transform=ax.transAxes, color="tab:red")

    ax.set_title(reference_name)
    ax.set_ylabel("match rate")
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="lower right")

axes[-1].set_xlabel("model version")
axes[-1].set_xticks(range(len(MODEL_VERSION_ORDER)), x_labels, rotation=45, ha="right")
fig.suptitle("OLMo2 ProFLingo Match Rate Relative to Each Reference", y=1.0)
fig.tight_layout()
plt.show()

## Match-Rate Table

In [ ]:
pivot = (
    match_df.pivot(index="model_version", columns="reference", values="match_rate")
    .reindex([label for _, label in MODEL_VERSION_ORDER])
)
pivot

## Optional Base Reference

When `olmo2_base_proflingo_reference_trajectory.json` is made compatible with the same summary format, add it to `REFERENCE_REPORTS` above. The cell below is a quick smoke check for whether the optional reports expose all requested target match rates.

In [ ]:
for reference_name, report_path in OPTIONAL_REFERENCE_REPORTS.items():
    if not report_path.exists():
        print(f"{reference_name}: missing {report_path}")
        continue
    optional_rows = pd.DataFrame(extract_proflingo_match_rates(reference_name, report_path))
    usable_rows = optional_rows["match_rate"].notna().sum()
    print(f"{reference_name}: {usable_rows}/{len(optional_rows)} requested targets have match_rate")